# Final annotated dataset

**Developed by:** Anna Maguza\
**Affiliation:** Faculty of Medicine, Würzburg University\
**Creation date:** 3rd March 2024\
**Last modified date:** 3rd March 2024

After the annotation of cell states, we wanted need to creare final annoted dataset which will contain all cells and be used for further analysis. For this we will concatenate cell type datasets created in previous steps, and fix the mismatches between cell type -- cell states. 

## Import packages

In [1]:
import numpy as np
import scanpy as sc
import anndata
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import json

## Upload data

In [2]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi = 180, color_map = 'magma_r', dpi_save = 300, vector_friendly = True, format = 'svg')

In [3]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

In [4]:
timestamp

'03032025_141544'

## Upload all datasets

In [5]:
adata = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_scVI_scANVI_celltypes_AM_10012025_100142_raw.h5ad')

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [6]:
adata_epi = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_epithelial_annotated_AM_29012025_122312_raw.h5ad')

In [7]:
adata_mes = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_mesenchymal_annotated_AM_02022025_163814_raw.h5ad')

In [8]:
adata_immune = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_immune_annotated_AM_18022025_185142_raw.h5ad')

In [9]:
adata_endo = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_scVI_scANVI_endothelial_cellstates_AM_28022025_102008_raw.h5ad')

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [10]:
adata_endo.obs['cell_states'] = adata_endo.obs['cellstates_scANVI'].copy()

In [11]:
adata_endo.obs.index = adata_endo.obs.index.astype(str)
adata_endo.obs_names_make_unique()

In [12]:
adata_neuronal = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_scVI_scANVI_neuronal_cellstates_AM_03022025_184954_raw.h5ad')

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [13]:
adata_neuronal.obs['cell_states'] = adata_neuronal.obs['cellstates_scANVI'].copy()

In [14]:
adata_neuronal.obs.index = adata_neuronal.obs.index.astype(str)
adata_neuronal.obs_names_make_unique()

### Merge datasets

In [15]:
anndata_objects = {'Epithelial': adata_epi, 'Mesenchymal': adata_mes, 'Immune': adata_immune, 'Neuronal': adata_neuronal, 'Endothelial': adata_endo}

In [16]:
merged_data = anndata.concat(
    list(anndata_objects.values()), 
    join='outer', 
    label=None, 
    keys=list(anndata_objects.keys()), 
    index_unique=None, 
    uns_merge='unique',
)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [17]:
duplicated_obs = merged_data.obs.index.duplicated().sum()
print(f"Number of duplicated observation indexes: {duplicated_obs}")

Number of duplicated observation indexes: 228


In [18]:
merged_data.obs.index = merged_data.obs.index.astype(str)
merged_data.obs['celltype'] = merged_data.obs['celltype'].astype(str)
merged_data.obs.index = merged_data.obs.index + '_' + merged_data.obs['celltype']
duplicated_obs = merged_data.obs.index.duplicated().sum()
print(f"Number of duplicated observation indexes: {duplicated_obs}")

Number of duplicated observation indexes: 0


+ Add processing history

In [19]:
current_history = adata.uns['processing_history'].tolist()

In [20]:
mesenchymal_entry1 = json.dumps({
    "timestamp": "30012025_144410", 
    "step": "Extracted genes (7000, library_preparation_protocol - batch) that were previously captured as highly variable, merged small populations into one group, fixed bolean nan issue, deleted samples with less than 10 cells, deleted doublets, predicted mesenchymal cell states using scVI-scANVI, scvi params: batch = sample_id, library_construnction_and_layout,Performer and Protocol REF, n_latent = 150, n_hidden = 256, n_layers = 3, dropout_rate = 0.1, dispersion = gene-cell, gene_likelihood = nb, 250 epochs, scanvi params: 300 epochs"
})
current_history.append(f"[Mesenchymal] {mesenchymal_entry1}")

mesenchymal_entry2 = json.dumps({
    "timestamp": "02022025_163814", 
    "step": "Manually annotated cell states"
})
current_history.append(f"[Mesenchymal] {mesenchymal_entry2}")

# Add Epithelial entries
epithelial_entry1 = json.dumps({
    "timestamp": "27012025_133833", 
    "step": "Extracted genes (7000, library_preparation_protocol - batch) that were previously captured as highly variable, merged EEC smaller populations into one group, fixed bolean nan issue, deleted samples with less than 10 cells, deleted doublets, predicted epithelial cell states using scVI-scANVI, scvi params: batch = sample_id, library_construnction_and_layout,Performer and Protocol REF, n_latent = 150, n_hidden = 256, n_layers = 3, dropout_rate = 0.1, dispersion = gene-cell, gene_likelihood = nb, 152 epochs, scanvi params: 462 epochs, concatenated small EECs substates into one group, predicted epithelial cell states using scVI-scANVI, scvi params: batch = sample_id, library_construnction_and_layout,Performer and Protocol REF, n_latent = 150, n_hidden = 256, n_layers = 3, dropout_rate = 0.1, dispersion = gene-cell, gene_likelihood = nb, 20 epochs, scanvi params: 300 epochs"
})
current_history.append(f"[Epithelial] {epithelial_entry1}")

epithelial_entry2 = json.dumps({
    "timestamp": "29012025_122312", 
    "step": "Manually annotated cell states"
})
current_history.append(f"[Epithelial] {epithelial_entry2}")

# Add Immune entries
immune_entry1 = json.dumps({
    "timestamp": "04022025_182820", 
    "step": "Extracted genes (7000, library_preparation_protocol - batch) that were previously captured as highly variable, merged small populations into one group, fixed bolean nan issue, deleted samples with less than 10 cells, deleted doublets, predicted immune cell states using scVI-scANVI, scvi params: batch = sample_id, library_construnction_and_layout,Performer and Protocol REF, n_latent = 150, n_hidden = 256, n_layers = 3, dropout_rate = 0.1, dispersion = gene-cell, gene_likelihood = nb, 300 epochs, scanvi params: 300 epochs"
})
current_history.append(f"[Immune] {immune_entry1}")

immune_entry2 = json.dumps({
    "timestamp": "18022025_185142", 
    "step": "manually annotated all immune cells"
})
current_history.append(f"[Immune] {immune_entry2}")

# Add Neuronal entry
neuronal_entry = json.dumps({
    "timestamp": "03022025_184954", 
    "step": "Extracted genes (7000, library_preparation_protocol - batch) that were previously captured as highly variable, merged small populations into one group, fixed bolean nan issue, deleted samples with less than 10 cells, deleted doublets, predicted neuronal cell states using scVI-scANVI, scvi params: batch = sample_id, library_construnction_and_layout,Performer and Protocol REF, n_latent = 150, n_hidden = 256, n_layers = 3, dropout_rate = 0.1, dispersion = gene-cell, gene_likelihood = nb, 115 epochs, scanvi params: 46 epochs"
})
current_history.append(f"[Neuronal] {neuronal_entry}")

# Add Endothelial entry
endothelial_entry = json.dumps({
    "timestamp": "28022025_102008", 
    "step": "Extracted genes (1500, library_preparation_protocol - batch. using old reference), merged small populations into one group, fixed bolean nan issue, deleted samples with less than 10 cells, deleted doublets, predicted endothelial cell states using scVI-scANVI, scvi params: batch = sample_id, library_construnction_and_layout,Performer and Protocol REF, n_latent = 50, n_hidden = 128, n_layers = 2, dropout_rate = 0.1, dispersion = gene-batch, gene_likelihood = nb, 100 epochs, scanvi params: 150 epochs"
})
current_history.append(f"[Endothelial] {endothelial_entry}")

# Add merged annotation entry
merged_entry = json.dumps({
    "timestamp": timestamp,
    "step": "Merged Epithelial, Immune, Mesenchymal, Neuronal, Endothelial cell state annotations into the main dataset"
})
current_history.append(merged_entry)

In [21]:
adata.uns['processing_history'] = current_history

In [22]:
adata.uns['processing_history']

['[E-MTAB-9720] {"step": "create raw anndata after mapping, no filtering", "timestamp": "06112024_121025"}',
 '[E-MTAB-9720] {"timestamp": "27112024_160858", "step": "added qc metrics (generated with sctk), doublets info, filtered cells with less than 100 genes"}',
 '[E-MTAB-9720] {"timestamp": "09122024_103213", "step": "filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and cell cycle phase"}',
 '[E-MTAB-9720] {"timestamp": "17122024_164056", "step": "Metadata harmonized for integration with other datasets"}',
 '[E-MTAB-9720] {"timestamp": "18122024_124621", "step": "Add metadata clusters using sentence-transformers"}',
 '[E-MTAB-E8901] {"step": "added qc metrics (generated with sctk), doublets info, filtered cells with less than 100 genes", "timestamp": "25112024_121501"}',
 '[E-MTAB-E8901] {"timestamp": "02122024_143702", "step": "filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and 

In [23]:
merged_data.uns['processing_history'] = adata.uns['processing_history'].copy()

### Fix cell state - cell type mismatches

In [24]:
unique_cell_types = merged_data.obs['celltype'].unique()
unique_cell_states = merged_data.obs['cell_states'].dropna().unique()

cell_state_to_cell_types = {}

for cell_state in unique_cell_states:
    cell_types_with_this_state = merged_data.obs.loc[merged_data.obs['cell_states'] == cell_state, 'celltype'].unique()
    cell_state_to_cell_types[cell_state] = cell_types_with_this_state

problematic_cell_states = {
    cell_state: cell_types.tolist() 
    for cell_state, cell_types in cell_state_to_cell_types.items() 
    if len(cell_types) > 1
}

if problematic_cell_states:
    print("Cell states appearing in multiple cell types:")
    for cell_state, cell_types in problematic_cell_states.items():
        print(f"  '{cell_state}' appears in: {', '.join(cell_types)}")
else:
    print("No cell states appear in multiple cell types - everything looks good!")

Cell states appearing in multiple cell types:
  'Lymphoid_Stroma' appears in: Mesenchymal, Myeloid, T cells, Plasma cells
  'cDC2' appears in: Myeloid, B cells
  'CLP' appears in: B cells, T cells
  'ILC3' appears in: T cells, B cells
  'Mast cells' appears in: Myeloid, B cells
  'Macrophages' appears in: Myeloid, T cells, B cells
  'Tfh' appears in: T cells, B cells
  'gamma delta T cells' appears in: T cells, B cells
  'IgA plasma cell' appears in: Myeloid, B cells, Plasma cells
  'Activated T' appears in: T cells, B cells
  'IgM plasma cell' appears in: Plasma cells, B cells
  'Naive CD8 T cells' appears in: T cells, B cells
  'Activated CD8 T' appears in: T cells, B cells
  'Activated CD4 T' appears in: T cells, B cells
  'Plasma cells' appears in: Plasma cells, B cells
  'Naive CD4 T cells' appears in: T cells, B cells
  'NK T cell' appears in: T cells, Myeloid
  'Treg' appears in: T cells, B cells
  'RBC' appears in: T cells, Myeloid


In [25]:
# Define the cell state to cell type mapping
cell_state_to_celltype = {
    'Lymphoid_Stroma': 'Mesenchymal',
    'cDC2': 'Myeloid',
    'ILC3': 'T cells',
    'Mast cells': 'Myeloid',
    'Macrophages': 'Myeloid',
    'Tfh': 'T cells',
    'gamma delta T cells': 'T cells',
    'IgA plasma cell': 'Plasma cells',
    'Activated T': 'T cells',
    'IgM plasma cell': 'Plasma cells',
    'Naive CD8 T cells': 'T cells',
    'Activated CD8 T': 'T cells',
    'Activated CD4 T': 'T cells',
    'Plasma cells': 'Plasma cells',
    'Naive CD4 T cells': 'T cells',
    'NK T cell': 'T cells',
    'Treg': 'T cells'
}

In [26]:
merged_data.obs['celltype'] = merged_data.obs['cell_states'].map(cell_state_to_celltype).fillna(merged_data.obs['celltype'])

In [29]:
merged_data = merged_data[~merged_data.obs['cell_states'].isin(['RBC'])]

### Add some more obs modifications

In [27]:
region_map = {
    'duodenum': 'small intestine',
    'ileum': 'small intestine',
    'small intestine': 'small intestine',
    'terminal ileum': 'small intestine',
    'jejunum': 'small intestine',
    'sigmoid colon': 'large intestine',
    'caecum': 'large intestine',
    'ascending colon': 'large intestine',
    'transverse colon': 'large intestine',
    'large intestine': 'large intestine',
    'descending colon': 'large intestine',
    'appendix': 'large intestine',
    'colon': 'large intestine',
    'rectum': 'large intestine'
}

In [28]:
merged_data.obs['gut_region'] = merged_data.obs['organism_part'].map(region_map)

### Save dataset

In [30]:
merged_data.obs['time'] = merged_data.obs['time'].astype('category')
merged_data.obs['time'] = merged_data.obs['time'].fillna('N/A')

/var/folders/kr/nd4y_1_s34n42lrht8wdh4cr0000gq/T/ipykernel_5103/1698762276.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  merged_data.obs['time'] = merged_data.obs['time'].astype('category')


In [31]:
merged_data.obs['time'] = merged_data.obs['time'].astype('str')

In [32]:
project = 'gut'
species = 'hs'
name = 'AM'
counts = 'raw'
atribute = 'all_datasets_full_annotated'

merged_data.write_h5ad(f"data/gut_data/{project}_{species}_{atribute}_{name}_{timestamp}_{counts}.h5ad")